# Qwen2.5-VL Step-DPO Fine-Tuning (Kaggle GPU)

This notebook trains a QLoRA adapter on the extracted Step-DPO pairs using TRL's `DPOTrainer` to align the model's step-by-step reasoning logic.

In [ ]:
# === Cell 1: Environment Setup & Hardware Safety Check ===
import os, sys, torch

# Load HF_TOKEN from Kaggle secrets if available
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets.')
except Exception:
    print('Kaggle secrets unavailable, skipping HF_TOKEN.')

# Check CUDA compute capability
print(f'Initial PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {props.name}, Compute Capability: {cc}, VRAM: {props.total_memory / 1e9:.1f} GB')
    if cc[0] < 7:
        print(f'*** Tesla P100 (cc {cc}) detected. PyTorch 2.10 dropped sm_60 CUDA kernels.')
        print('*** Installing PyTorch 2.5.1+cu124 with full sm_60 CUDA GPU support...')
        os.system('pip install -q "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu124')
else:
    print('CUDA is not available.')

# Clone project repo to /tmp to keep /kaggle/working clean
repo_dir = '/tmp/prm_project'
if not os.path.exists(repo_dir):
    !git clone https://github.com/yahorlahunovich/prm_project.git {repo_dir}
else:
    !git -C {repo_dir} pull --ff-only

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
os.chdir(repo_dir)

In [ ]:
# === Cell 2: Install Dependencies & Remove Incompatible torchao ===
# Remove outdated pre-installed torchao to prevent PEFT ImportError
!pip uninstall -y torchao

!pip install -q \
    "transformers>=4.49.0" \
    "trl>=0.12.0" \
    "peft>=0.10.0" \
    "accelerate>=0.30.0" \
    "datasets" \
    "qwen-vl-utils"

import torch, transformers, trl, peft, accelerate
print(f'Active PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU device: {torch.cuda.get_device_name(0)}')
print(f'transformers: {transformers.__version__}')
print(f'trl: {trl.__version__}')
print(f'peft: {peft.__version__}')
print(f'accelerate: {accelerate.__version__}')

In [ ]:
# === Cell 3: Load Dataset ===
import json, os, torch
from datasets import Dataset
from PIL import Image

# Ensure chart images are downloaded
images_dir = 'data/CharXiv/images'
if not os.path.exists(images_dir) or len(os.listdir(images_dir)) == 0:
    print('Downloading chart images...')
    os.system('python scripts/download_images.py')

# Load Step-DPO pairs
data_path = 'experiments/001_500_reasoning/data/step_dpo_pairs.jsonl'
with open(data_path) as f:
    raw_data = [json.loads(line) for line in f]

hf_data = {'prompt': [], 'chosen': [], 'rejected': [], 'images': []}
skipped = 0

for item in raw_data:
    img_path = os.path.abspath(item['image_path'])
    try:
        img = Image.open(img_path).convert('RGB')
    except Exception as e:
        skipped += 1
        continue

    question = item.get('question', '').strip()
    prompt_text = f'Analyze this chart. Provide step-by-step reasoning and a final answer.\n{question}'

    prompt_msg = [{
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': prompt_text}
        ]
    }]

    prefix = item.get('prefix', '')
    chosen_msg = [{'role': 'assistant', 'content': prefix + item['chosen']}]
    rejected_msg = [{'role': 'assistant', 'content': prefix + item['rejected']}]

    hf_data['prompt'].append(prompt_msg)
    hf_data['chosen'].append(chosen_msg)
    hf_data['rejected'].append(rejected_msg)
    hf_data['images'].append([img])

dataset = Dataset.from_dict(hf_data)
print(f'Loaded {len(dataset)} Step-DPO pairs (skipped {skipped} due to missing images).')
print(dataset)

In [ ]:
# === Cell 4: Load Model & Processor ===
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

model_id = 'Qwen/Qwen2.5-VL-3B-Instruct'
is_gpu = torch.cuda.is_available()
device_map = {'': 0} if is_gpu else 'cpu'

print(f'Loading model with device_map={device_map} (is_gpu={is_gpu})...')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    attn_implementation='sdpa' if is_gpu else 'eager',
    device_map=device_map,
)
model.enable_input_require_grads()

# Freeze vision encoder — only train language model LoRA
if hasattr(model, 'visual'):
    model.visual.requires_grad_(False)

processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# Expose token properties directly on processor for DPOTrainer
processor.pad_token = processor.tokenizer.pad_token
processor.pad_token_id = processor.tokenizer.pad_token_id
processor.eos_token_id = processor.tokenizer.eos_token_id

print(f'Model loaded on device: {next(model.parameters()).device}')

In [ ]:
# === Cell 5: Configure LoRA & DPO Trainer ===
import os, torch
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM',
)

is_gpu = torch.cuda.is_available()
out_dir = '/kaggle/working/dpo_qwen_vl' if os.path.exists('/kaggle/working') else './dpo_qwen_vl'

training_args = DPOConfig(
    output_dir=out_dir,
    beta=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=3,
    max_length=2048,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    gradient_checkpointing=is_gpu,
    dataset_num_proc=1,
    remove_unused_columns=False,
    report_to='none',
    fp16=is_gpu,
    bf16=False,
)

print('Initializing DPOTrainer...')
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=dataset,
    processing_class=processor,
    peft_config=peft_config,
)
print('DPOTrainer initialized successfully.')

In [ ]:
# === Cell 6: Train & Save ===
print('Starting DPO training...')
trainer.train()
save_path = '/kaggle/working/qwen_vl_step_dpo_adapter' if os.path.exists('/kaggle/working') else 'qwen_vl_step_dpo_adapter'
trainer.save_model(save_path)
print(f'\nTraining complete! Adapter saved to {save_path}')